# Notebook 6 — Consolidación de revisión manual de entidades

Este notebook toma el Excel revisado manualmente y el JSON simplificado anterior, y genera un nuevo JSON/CSV con las entidades corregidas, borradas y actualizadas.

In [ ]:
# Celda 1 — Imports y rutas
# En Colab, subí estos dos archivos:
# - embargos_revision_entidades_REVISION2.xlsx
# - embargos_revision_entidades_estructura.json

import json
from pathlib import Path
from collections import Counter

import pandas as pd

EXCEL_PATH = Path("/content/embargos_revision_entidades_REVISION2.xlsx")
JSON_BASE_PATH = Path("/content/embargos_revision_entidades_estructura.json")

SHEET_CANDIDATES = [
    "revision_entidades_actualizadas_borradas",
    "revision_entidades_actualizadas",  # Excel suele cortar nombres de hoja largos a 31 caracteres
    "revision_entidades_actualizado",
    "revisador",
]

OUTPUT_JSON = Path("/content/embargos_revision_entidades_actualizado.json")
OUTPUT_CSV = Path("/content/embargos_revision_entidades_actualizado.csv")

## Celda 2 — Cargar archivos

Se carga el Excel revisado y el JSON base. La hoja final revisada es la fuente de verdad: las filas borradas no pasan al JSON nuevo, y los valores editados reemplazan los valores anteriores.

In [ ]:
# Si estás en Colab y todavía no subiste los archivos, descomentá esto:
# from google.colab import files
# files.upload()

xls = pd.ExcelFile(EXCEL_PATH)
print("Hojas encontradas:", xls.sheet_names)

sheet_name = next((s for s in SHEET_CANDIDATES if s in xls.sheet_names), None)
if sheet_name is None:
    raise ValueError(f"No encontré la hoja revisada. Hojas disponibles: {xls.sheet_names}")

print("Hoja usada:", sheet_name)

df_rev = pd.read_excel(EXCEL_PATH, sheet_name=sheet_name)

with open(JSON_BASE_PATH, "r", encoding="utf-8") as f:
    docs_base = json.load(f)

print("Filas en Excel revisado:", len(df_rev))
print("Documentos en JSON base:", len(docs_base))

df_rev.head()

## Celda 3 — Limpieza mínima de columnas

Normalizamos nombres de columnas y dejamos solo las filas que tienen entidad real: `numero_archivo`, `etiqueta` y `valor`. Además, si alguna fila quedó como `revision manual`, se normaliza a `manual`.

In [ ]:
df_rev.columns = [str(c).strip() for c in df_rev.columns]

required_cols = ["numero_archivo", "id", "nombre_archivo", "cantidad_entidades_encontradas", "etiqueta", "valor", "metodo"]
missing = [c for c in required_cols if c not in df_rev.columns]
if missing:
    raise ValueError(f"Faltan columnas requeridas: {missing}. Columnas actuales: {df_rev.columns.tolist()}")

# Nos quedamos solo con entidades válidas
df_rev = df_rev.dropna(subset=["numero_archivo", "etiqueta", "valor"]).copy()

df_rev["numero_archivo"] = df_rev["numero_archivo"].astype(int)

# Normalizar método manual
# Aclaración: el nombre correcto del método corregido a mano es "manual".
df_rev["metodo"] = df_rev["metodo"].astype(str).str.strip()
df_rev["metodo"] = df_rev["metodo"].replace({
    "revision manual": "manual",
    "revisión manual": "manual",
    "Revision manual": "manual",
    "Revisión manual": "manual",
})

print("Filas válidas luego de limpieza:", len(df_rev))
print(df_rev["metodo"].value_counts())

## Celda 4 — Preparar datos base y funciones auxiliares

El Excel revisado no siempre conserva `texto_limpio`, `clasificacion`, `span_inicio` y `span_fin`. Por eso se recuperan desde el JSON base. Para los spans, se hace una asociación por orden y etiqueta dentro de cada archivo.

In [ ]:
docs_base_by_num = {int(doc["numero_archivo"]): doc for doc in docs_base}


def clean_value(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    return s if s else None


def clean_int_or_none(x):
    if x is None or pd.isna(x):
        return None
    try:
        return int(float(x))
    except Exception:
        return None


def assign_spans(numero_archivo, revised_entities):
    """
    Recupera span_inicio y span_fin desde el JSON base.
    Como algunas entidades fueron editadas o borradas, se hace matching principalmente por orden y etiqueta.
    """
    original_entities = docs_base_by_num.get(numero_archivo, {}).get("entidades", [])
    used = set()
    pointer = 0
    output = []

    for ent in revised_entities:
        etiqueta = ent["etiqueta"]
        valor = ent["valor"]
        metodo = ent["metodo"]
        chosen = None

        # 1) Match exacto etiqueta + valor + método
        for i in range(pointer, len(original_entities)):
            orig = original_entities[i]
            if i in used:
                continue
            if (
                orig.get("etiqueta") == etiqueta
                and str(orig.get("valor", "")).strip() == valor
                and str(orig.get("metodo", "")).strip() == metodo
            ):
                chosen = i
                break

        # 2) Match exacto etiqueta + valor
        if chosen is None:
            for i in range(pointer, len(original_entities)):
                orig = original_entities[i]
                if i in used:
                    continue
                if orig.get("etiqueta") == etiqueta and str(orig.get("valor", "")).strip() == valor:
                    chosen = i
                    break

        # 3) Misma etiqueta por orden, útil cuando el valor fue corregido manualmente
        if chosen is None:
            for i in range(pointer, len(original_entities)):
                orig = original_entities[i]
                if i in used:
                    continue
                if orig.get("etiqueta") == etiqueta:
                    chosen = i
                    break

        span_inicio = None
        span_fin = None
        if chosen is not None:
            used.add(chosen)
            pointer = max(pointer, chosen + 1)
            span_inicio = original_entities[chosen].get("span_inicio")
            span_fin = original_entities[chosen].get("span_fin")

        output.append({
            "etiqueta": etiqueta,
            "valor": valor,
            "metodo": metodo,
            "span_inicio": clean_int_or_none(span_inicio),
            "span_fin": clean_int_or_none(span_fin),
        })

    return output

## Celda 5 — Reconstruir el JSON actualizado

Se agrupa el Excel por `numero_archivo`. La cantidad de entidades se recalcula luego de la revisión manual, por eso refleja entidades borradas y conservadas.

In [ ]:
json_actualizado = []

for numero_archivo, group in df_rev.groupby("numero_archivo", sort=True):
    base_doc = docs_base_by_num.get(int(numero_archivo), {})
    first = group.iloc[0]

    revised_entities = []
    for _, row in group.iterrows():
        revised_entities.append({
            "etiqueta": clean_value(row["etiqueta"]),
            "valor": clean_value(row["valor"]),
            "metodo": clean_value(row["metodo"]),
        })

    entidades = assign_spans(int(numero_archivo), revised_entities)

    json_actualizado.append({
        "numero_archivo": int(numero_archivo),
        "id": clean_value(first["id"]) or base_doc.get("id"),
        "nombre_archivo": clean_value(first["nombre_archivo"]) or base_doc.get("nombre_archivo"),
        "clasificacion": base_doc.get("clasificacion"),
        "texto_limpio": base_doc.get("texto_limpio"),
        "cantidad_entidades_encontradas": len(entidades),
        "entidades": entidades,
    })

print("Documentos actualizados:", len(json_actualizado))
print("Entidades actualizadas:", sum(doc["cantidad_entidades_encontradas"] for doc in json_actualizado))

## Celda 6 — Guardar JSON y CSV

El JSON mantiene la estructura anidada por archivo. El CSV queda aplanado: una fila por entidad, para poder revisarlo o cruzarlo fácilmente.

In [ ]:
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(json_actualizado, f, ensure_ascii=False, indent=2)

rows = []
for doc in json_actualizado:
    for ent in doc["entidades"]:
        rows.append({
            "numero_archivo": doc["numero_archivo"],
            "id": doc["id"],
            "nombre_archivo": doc["nombre_archivo"],
            "clasificacion": doc["clasificacion"],
            "texto_limpio": doc["texto_limpio"],
            "cantidad_entidades_encontradas": doc["cantidad_entidades_encontradas"],
            "etiqueta": ent["etiqueta"],
            "valor": ent["valor"],
            "metodo": ent["metodo"],
            "span_inicio": ent["span_inicio"],
            "span_fin": ent["span_fin"],
        })

df_csv = pd.DataFrame(rows)
df_csv.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("JSON guardado:", OUTPUT_JSON)
print("CSV guardado:", OUTPUT_CSV)

## Celda 7 — Validaciones rápidas

Estas validaciones permiten revisar cuántas entidades quedaron por tipo y por método. También controla cuántas correcciones manuales quedaron como `manual`.

In [ ]:
print("Cantidad de documentos:", len(json_actualizado))
print("Total entidades:", sum(doc["cantidad_entidades_encontradas"] for doc in json_actualizado))

print("
Entidades por etiqueta:")
print(df_csv["etiqueta"].value_counts())

print("
Entidades por método:")
print(df_csv["metodo"].value_counts())

print("
Correcciones manuales:", (df_csv["metodo"] == "manual").sum())

df_csv.head()

## Celda 8 — Descargar archivos en Colab

In [ ]:
from google.colab import files

files.download(str(OUTPUT_JSON))
files.download(str(OUTPUT_CSV))